In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_excel("./data/Raisin_Dataset.xlsx")

In [3]:
df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
0,87524,442.246011,253.291155,0.819738,90546,0.758651,1184.040,Kecimen
1,75166,406.690687,243.032436,0.801805,78789,0.684130,1121.786,Kecimen
2,90856,442.267048,266.328318,0.798354,93717,0.637613,1208.575,Kecimen
3,45928,286.540559,208.760042,0.684989,47336,0.699599,844.162,Kecimen
4,79408,352.190770,290.827533,0.564011,81463,0.792772,1073.251,Kecimen


In [4]:
x = df.drop(["Area", "Class"], axis = 1)
y = df["Class"]

In [5]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, stratify = y,
                                                    random_state = 23)

In [6]:
mm = MinMaxScaler()

In [7]:
scaled_train = mm.fit_transform(x_train)
scaled_test = mm.transform(x_test)

In [8]:
# 최근접 이웃의 수
K = 5

In [9]:
scaled_train.shape

(630, 6)

In [11]:
scaled_test[0].shape

(6,)

In [14]:
# 예측 예시
print(scaled_train[[0, 1]])
print(scaled_test[0])
print(np.linalg.norm(scaled_train[[0, 1]] - scaled_test[0], axis = 1))

[[0.52583543 0.46227141 0.85537041 0.57472036 0.92706218 0.53550554]
 [0.16023647 0.32726027 0.50713682 0.20926585 0.81082668 0.21399459]]
[0.31813744 0.36943103 0.75062248 0.34477868 0.61720112 0.34253226]
[0.49885434 0.39796089]


In [15]:
# 예측
# 하나의 데이터와 모든 훈련데이터와의 거리
dist = np.linalg.norm(scaled_train - scaled_test[0], axis = 1)

In [16]:
dist.shape

(630,)

In [17]:
dist

array([0.49885434, 0.39796089, 0.27425311, 0.64416338, 0.4191368 ,
       0.45832072, 0.83293975, 0.27274125, 0.1901054 , 0.20282574,
       0.52353718, 0.34759694, 0.18502169, 0.37220821, 0.12189319,
       0.42713111, 0.52617401, 0.27012403, 0.29681878, 1.30153807,
       0.22137275, 0.38237489, 0.57283447, 0.35024764, 0.27632147,
       0.20642558, 0.24443717, 0.12995284, 0.29140858, 0.25620231,
       0.19612206, 0.42712428, 0.2780764 , 0.33022703, 0.62157581,
       0.2700429 , 0.11922156, 0.29755587, 0.64589859, 0.36420293,
       0.80210024, 0.21157845, 0.05262205, 0.43239108, 0.61540323,
       0.49928007, 0.37937327, 0.16724427, 0.21735635, 0.39918714,
       0.34215832, 0.15787987, 0.40686606, 0.61730346, 0.25742486,
       0.19265338, 0.2459929 , 0.37719884, 0.13664526, 0.30055322,
       0.22314069, 0.27506824, 0.22324236, 0.47835306, 0.20575262,
       0.21567278, 0.15524529, 0.36398715, 0.56689467, 0.11800518,
       0.33341557, 0.29031345, 0.15010924, 0.48833773, 0.25414

In [18]:
# 거리가 가장 가까운 데이터들의 인덱스
k_index = dist.argsort()[:K]
k_index

array([248, 454,  42, 551, 194], dtype=int64)

In [19]:
# 거리가 가장 가까운 데이터들의 레이블
k_nearest_label = y_train.values[k_index]

In [20]:
k_nearest_label

array(['Besni', 'Besni', 'Besni', 'Kecimen', 'Kecimen'], dtype=object)

In [21]:
most_common = Counter(k_nearest_label).most_common(1)
most_common

[('Besni', 3)]

In [22]:
# KNN 클래스 정의
class KNN:
    def __init__(self, k:int = 5):
        '''
        KNN 분류기 초기화

        Parameters:
        k : 최근접 이웃의 수
        '''
        self.k = k

    def fit(self, x:np.ndarray, y:np.ndarray):
        '''
        학습 데이터와 레이블을 저장

        Parameters:
        x : 학습 데이터, shape -> (num_samples, num_features)
        y : 레이블, shape -> (num_samples,)
        '''
        self.x_train = x
        self.y_train = y

    def predict(self, x:np.ndarray) -> np.ndarray:
        '''
        새로운 데이터 포인트에 대한 예측을 수행

        Parameters:
        x : 테스트 데이터, shape -> (num_test_samples, num_features)

        Returns:
        예측된 레이블, shape -> (num_test_samples,)
        '''
        predictions = []

        for i in x:
            # 모든 학습 데이터와의 거리 계산
            distances = np.linalg.norm(self.x_train - i, axis = 1)

            # 가장 가까운 k개의 이웃의 인덱스
            k_indexes = distances.argsort()[:self.k]

            # 가장 많이 등장하는 레이블을 예측
            k_nearest_labels = self.y_train[k_indexes]
            most_common = Counter(k_nearest_labels).most_common(1)
            predictions.append(most_common[0][0])

        return np.array(predictions)

In [23]:
knn = KNN(k = 5)

In [24]:
knn.fit(scaled_train, y_train.values)

In [25]:
knn.predict(scaled_test[[0]])

array(['Besni'], dtype='<U5')

In [26]:
knn.predict(scaled_test)

array(['Besni', 'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen',
       'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen',
       'Besni', 'Besni', 'Besni', 'Besni', 'Besni', 'Kecimen', 'Kecimen',
       'Besni', 'Besni', 'Besni', 'Kecimen', 'Besni', 'Besni', 'Besni',
       'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen',
       'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen',
       'Besni', 'Besni', 'Kecimen', 'Besni', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Besni',
       'Kecimen', 'Kecimen', 'Kecimen', 'Besni', 'Kecimen', 'Besni',
       'Kecimen', 'Kecimen', 'Kecimen', 'Besni', 'Kecimen', 'Besni',
       'Besni', 'Besni', 'Kecimen', 'Besni', 'Kecimen', 'Besni', 'Besni',
       'Besni', 'Kecimen', 'Kecimen', 'Besni', 'Kecimen', 'Kecimen',
       'Besni', 

In [27]:
# KNN 클래스 거리 가중치 추가
class KNN:
    def __init__(self, k:int = 5, weighted:bool = False):
        '''
        KNN 분류기 초기화

        Parameters:
        k : 최근접 이웃의 수
        '''
        self.k = k
        self.weighted = weighted

    def fit(self, x:np.ndarray, y:np.ndarray):
        '''
        학습 데이터와 레이블을 저장

        Parameters:
        x : 학습 데이터, shape -> (num_samples, num_features)
        y : 레이블, shape -> (num_samples,)
        '''
        self.x_train = x
        self.y_train = y

    def predict(self, x:np.ndarray) -> np.ndarray:
        '''
        새로운 데이터 포인트에 대한 예측을 수행

        Parameters:
        x : 테스트 데이터, shape -> (num_test_samples, num_features)

        Returns:
        예측된 레이블, shape -> (num_test_samples,)
        '''
        predictions = []

        for i in x:
            # 모든 학습 데이터와의 거리 계산
            distances = np.linalg.norm(self.x_train - i, axis = 1)

            # 가장 가까운 k개의 이웃의 인덱스
            k_indexes = distances.argsort()[:self.k]

            # 가장 많이 등장하는 레이블을 예측
            k_nearest_labels = self.y_train[k_indexes]

            if self.weighted:
                # 거리 가중치를 사용하는 경우(가까울수록 가중치 높음)
                # 거리에 작은 값을 더하여 0으로 나누어지는 경우를 방지
                weights = 1 / (distances[k_indexes] + 1e-5)

                label_weights = dict()
                for label, weight in zip(k_nearest_labels, weights):
                    label_weights[label] = label_weights.get(label, 0) + weight

                # 가장 높은 가중치를 가진 레이블 선택
                predicted_label = max(label_weights.items(), key = lambda item: item[1])[0]

            else:
                # 거리 가중치 미사용
                most_common = Counter(k_nearest_labels).most_common(1)
                predicted_label = most_common[0][0]

            predictions.append(predicted_label)

        return np.array(predictions)

In [28]:
knn = KNN(k = 5, weighted = True)

In [29]:
knn.fit(scaled_train, y_train.values)

In [30]:
knn.predict(scaled_test[[0]])

array(['Besni'], dtype='<U5')

In [31]:
knn.predict(scaled_test)

array(['Besni', 'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen',
       'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen',
       'Besni', 'Besni', 'Besni', 'Besni', 'Besni', 'Kecimen', 'Kecimen',
       'Besni', 'Besni', 'Besni', 'Kecimen', 'Besni', 'Besni', 'Besni',
       'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen',
       'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen',
       'Besni', 'Besni', 'Kecimen', 'Besni', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Besni',
       'Kecimen', 'Kecimen', 'Kecimen', 'Besni', 'Kecimen', 'Besni',
       'Kecimen', 'Kecimen', 'Kecimen', 'Besni', 'Kecimen', 'Besni',
       'Besni', 'Besni', 'Besni', 'Besni', 'Kecimen', 'Besni', 'Besni',
       'Besni', 'Kecimen', 'Kecimen', 'Besni', 'Besni', 'Kecimen',
       'Besni', 'Kec

In [34]:
(y_test == knn.predict(scaled_test)).mean()

0.8407407407407408

In [35]:
# KNN 클래스 거리 가중치 추가, 거리 측정 방식 추가
class KNN:
    def __init__(self, k:int = 5, weighted:bool = False, metric:str = "euclidean"):
        '''
        KNN 분류기 초기화

        Parameters:
        k : 최근접 이웃의 수
        '''
        self.k = k
        self.weighted = weighted
        self.metric = metric

    def distance(self, x1, x2):
        if self.metric == "euclidean":
            return np.sqrt(np.sum((x1 - x2) ** 2, axis = 1))

        elif self.metric == "manhattan":
            return np.sum(np.abs(x1 - x2), axis = 1)

        else:
            ValueError("지원하지 않는 거리 계산 방식입니다")

    def fit(self, x:np.ndarray, y:np.ndarray):
        '''
        학습 데이터와 레이블을 저장

        Parameters:
        x : 학습 데이터, shape -> (num_samples, num_features)
        y : 레이블, shape -> (num_samples,)
        '''
        self.x_train = x
        self.y_train = y

    def predict(self, x:np.ndarray) -> np.ndarray:
        '''
        새로운 데이터 포인트에 대한 예측을 수행

        Parameters:
        x : 테스트 데이터, shape -> (num_test_samples, num_features)

        Returns:
        예측된 레이블, shape -> (num_test_samples,)
        '''
        predictions = []

        for i in x:
            # 모든 학습 데이터와의 거리 계산
            distances = self.distance(self.x_train, i)

            # 가장 가까운 k개의 이웃의 인덱스
            k_indexes = distances.argsort()[:self.k]

            # 가장 많이 등장하는 레이블을 예측
            k_nearest_labels = self.y_train[k_indexes]

            if self.weighted:
                # 거리 가중치를 사용하는 경우(가까울수록 가중치 높음)
                # 거리에 작은 값을 더하여 0으로 나누어지는 경우를 방지
                weights = 1 / (distances[k_indexes] + 1e-5)

                label_weights = dict()
                for label, weight in zip(k_nearest_labels, weights):
                    label_weights[label] = label_weights.get(label, 0) + weight

                # 가장 높은 가중치를 가진 레이블 선택
                predicted_label = max(label_weights.items(), key = lambda item: item[1])[0]

            else:
                # 거리 가중치 미사용
                most_common = Counter(k_nearest_labels).most_common(1)
                predicted_label = most_common[0][0]

            predictions.append(predicted_label)

        return np.array(predictions)

In [36]:
# K = 5, 거리 가중치 사용, 맨하탄 거리 사용
knn = KNN(k = 5, weighted = True, metric = "manhattan")

In [37]:
knn.fit(scaled_train, y_train.values)

In [38]:
knn.predict(scaled_test[[0]])

array(['Besni'], dtype='<U5')

In [39]:
knn.predict(scaled_test)

array(['Besni', 'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen',
       'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen',
       'Besni', 'Besni', 'Besni', 'Besni', 'Besni', 'Kecimen', 'Kecimen',
       'Besni', 'Kecimen', 'Besni', 'Kecimen', 'Besni', 'Besni',
       'Kecimen', 'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Besni',
       'Besni', 'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Kecimen',
       'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen',
       'Kecimen', 'Besni', 'Besni', 'Kecimen', 'Besni', 'Kecimen',
       'Besni', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen', 'Kecimen',
       'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen', 'Besni',
       'Kecimen', 'Besni', 'Kecimen', 'Kecimen', 'Kecimen', 'Besni',
       'Kecimen', 'Besni', 'Besni', 'Besni', 'Besni', 'Besni', 'Kecimen',
       'Besni', 'Besni', 'Besni', 'Kecimen', 'Kecimen', 'Besni',
       'Kecimen', 'Kecimen', 'Besni'

In [40]:
(y_test == knn.predict(scaled_test)).mean()

0.8481481481481481